# Hebrew Letterform Classifier (Phase 4) — Kaggle GPU

Trains a 27-class CNN on the Phase-1 synthetic glyph shards.

**Before running:**
1. Generate data locally: `python src/synth_data.py`
2. Upload `data/synth/` as a Kaggle Dataset (e.g. slug `hebrew-synth`).
3. Notebook: **Add Data** -> your dataset, then **Settings -> Accelerator -> GPU**.
4. Set `DATA` below to the mount path, then **Run All**.

Outputs (`classifier.pth`, `metrics.json`, `confusion_matrix.png`) land in `/kaggle/working` — download from the **Output** tab.

In [ ]:
!pip install -q webdataset


In [ ]:
# ── config ─────────────────────────────────────────────────────────────
import glob, os
# Point DATA at the folder that holds glyphs-*.tar + classes.json.
# Find it from the right panel; usually /kaggle/input/<your-dataset-slug>
DATA = '/kaggle/input/hebrew-synth/synth'
if not glob.glob(os.path.join(DATA, 'glyphs-*.tar')):
    cand = glob.glob('/kaggle/input/**/glyphs-*.tar', recursive=True)
    assert cand, 'No glyphs-*.tar found under /kaggle/input — did you add the dataset?'
    DATA = os.path.dirname(cand[0])
OUT = '/kaggle/working'
EPOCHS, BATCH, LR, VAL_SPLIT, IMG, SEED = 25, 128, 1e-3, 0.1, 64, 1234
print('DATA =', DATA)


In [ ]:
# ── load shards into memory ────────────────────────────────────────────
import json, numpy as np, webdataset as wds
from PIL import Image

shards = sorted(glob.glob(os.path.join(DATA, 'glyphs-*.tar')))
classes = None
cp = os.path.join(DATA, 'classes.json')
if os.path.exists(cp): classes = json.load(open(cp))['alphabet']
X, y = [], []
for s in wds.WebDataset(shards, shardshuffle=False).decode('pil'):
    a = np.asarray(s['png'].convert('L'))
    if a.shape != (IMG, IMG): a = np.asarray(Image.fromarray(a).resize((IMG, IMG)))
    X.append(a); y.append(int(s['cls']))
X = np.stack(X); y = np.asarray(y, np.int64)
num_classes = len(classes) if classes else int(y.max())+1
print('glyphs:', len(X), ' classes:', num_classes)


In [ ]:
# ── model ──────────────────────────────────────────────────────────────
import torch, torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset, random_split
torch.manual_seed(SEED)
device = 'cuda' if torch.cuda.is_available() else 'cpu'; print('device:', device)

def block(ci, co):
    return nn.Sequential(nn.Conv2d(ci,co,3,padding=1), nn.BatchNorm2d(co), nn.ReLU(True),
                         nn.Conv2d(co,co,3,padding=1), nn.BatchNorm2d(co), nn.ReLU(True), nn.MaxPool2d(2))
model = nn.Sequential(block(1,32), block(32,64), block(64,128),
                      nn.AdaptiveAvgPool2d(1), nn.Flatten(),
                      nn.Linear(128,256), nn.ReLU(True), nn.Dropout(0.4),
                      nn.Linear(256,num_classes)).to(device)


In [ ]:
# ── train ──────────────────────────────────────────────────────────────
Xt = torch.from_numpy(X).float().div_(255).unsqueeze(1)
yt = torch.from_numpy(y)
full = TensorDataset(Xt, yt)
n_val = int(len(full)*VAL_SPLIT)
g = torch.Generator().manual_seed(SEED)
tr, va = random_split(full, [len(full)-n_val, n_val], generator=g)
tr_dl = DataLoader(tr, batch_size=BATCH, shuffle=True, num_workers=2)
va_dl = DataLoader(va, batch_size=BATCH, num_workers=2)

opt = torch.optim.Adam(model.parameters(), lr=LR)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, EPOCHS)
crit = nn.CrossEntropyLoss()
best_acc, best_cm = 0.0, None
for ep in range(EPOCHS):
    model.train(); tot=0.0
    for xb,yb in tr_dl:
        xb,yb = xb.to(device), yb.to(device)
        opt.zero_grad(); loss = crit(model(xb), yb); loss.backward(); opt.step()
        tot += loss.item()*len(xb)
    sched.step()
    model.eval(); correct=0; cm=np.zeros((num_classes,num_classes),np.int64)
    with torch.no_grad():
        for xb,yb in va_dl:
            p = model(xb.to(device)).argmax(1).cpu()
            correct += (p==yb).sum().item()
            for t,q in zip(yb.numpy(), p.numpy()): cm[t,q]+=1
    acc = correct/max(len(va),1)
    print(f'epoch {ep+1:2d}/{EPOCHS}  train_loss={tot/len(tr):.4f}  val_acc={acc:.4f}')
    if acc >= best_acc:
        best_acc, best_cm = acc, cm
        torch.save({'state_dict':model.state_dict(),'num_classes':num_classes,'classes':classes,'img':IMG}, f'{OUT}/classifier.pth')
print('BEST val_acc =', best_acc)


In [ ]:
# ── confusion matrix + metrics ─────────────────────────────────────────
import matplotlib.pyplot as plt
json.dump({'val_acc':best_acc,'classes':classes}, open(f'{OUT}/metrics.json','w'), ensure_ascii=False, indent=2)
cmn = best_cm / best_cm.sum(1, keepdims=True).clip(min=1)
labels = classes or [str(i) for i in range(num_classes)]
fig, ax = plt.subplots(figsize=(10,9)); ax.imshow(cmn, cmap='viridis')
ax.set_xticks(range(num_classes)); ax.set_xticklabels(labels)
ax.set_yticks(range(num_classes)); ax.set_yticklabels(labels)
ax.set_xlabel('predicted'); ax.set_ylabel('true'); ax.set_title('Confusion (row-normalized)')
fig.tight_layout(); fig.savefig(f'{OUT}/confusion_matrix.png', dpi=120); plt.show()
print('saved ->', os.listdir(OUT))
